# Train the potato detector locally (Jupyter)

One-class YOLOv8n potato detector → ONNX, to run in-browser with onnxruntime-web.

**Before launching Jupyter:** activate your Python env first (`venv_pls`), so these cells
use the right interpreter.

Differences from the Colab version: no `google.colab` download cell — the trained model
is already on disk here, so the last cell just copies it into the repo.

In [ ]:
# Cell 1 — install deps into the active env.
# (If you'd rather install from your terminal after `venv_pls`, run the same pip line there.)
%pip install ultralytics roboflow onnx onnxruntime

In [ ]:
# Cell 2 — PASTE YOUR ROBOFLOW SNIPPET HERE (the YOLOv8-format download you copied).
# It looks roughly like this — replace with your exact key/workspace/project/version:
from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_API_KEY")
project = rf.workspace("yolo-4qlka").project("potato-detection-3et6q-o4ogu")
version = project.version(1)
dataset = version.download("yolov8")

data_yaml = dataset.location + "/data.yaml"
print("data.yaml at:", data_yaml)

In [ ]:
# Cell 3 — train.
#   device: on Apple Silicon use 'mps' (Metal GPU, much faster). On an Intel Mac use 'cpu'.
#   If unsure, leave device=None and Ultralytics auto-picks; set 'mps' explicitly if it
#   falls back to CPU on Apple Silicon.
#   imgsz=320 keeps the browser model fast. CPU training will be slow — lower epochs if needed.
from ultralytics import YOLO

DEVICE = "mps"   # 'mps' (Apple Silicon) | 'cpu' (Intel) | None (auto)

model = YOLO("yolov8n.pt")
model.train(data=data_yaml, epochs=60, imgsz=320, batch=16, device=DEVICE)

In [ ]:
# Cell 4 — optional sanity check (mAP + class names).
best = YOLO("runs/detect/train/weights/best.pt")
metrics = best.val(device=DEVICE)
print("mAP50-95:", metrics.box.map)
print("classes:", best.names)   # tell me this exact name/order

In [ ]:
# Cell 5 — export to ONNX (opset 12 + simplify plays nicely with onnxruntime-web).
onnx_path = best.export(format="onnx", imgsz=320, opset=12, simplify=True)
print("exported:", onnx_path)   # -> runs/detect/train/weights/best.onnx

In [ ]:
# Cell 6 — copy the exported model into the repo where the rig will load it.
# Adjust REPO if this notebook isn't running from inside the repo.
import shutil, os
REPO = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))  # -> shmspace/ if run from potato_model/
dest = os.path.join(REPO, "public", "puppets", "potato_model", "potato.onnx")
shutil.copy(onnx_path, dest)
print("copied model to:", dest)
print("input size: 320  |  classes:", best.names)